In [1]:
import pandas as pd
import shap
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import joblib


/Users/mahsaamani/Downloads/Saarlanduni/DataScience/UnveilingHospitalCostDrivers/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')
data.head()

/var/folders/s1/z7p70y8n35lcdk9lq0wvfxz00000gq/T/ipykernel_17970/1113152288.py:2: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../data/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,50 to 69,107,F,White,Not Span/Hispanic,...,Major,Major,Medical,Medicaid,NaN,NaN,NaN,Y,"51,514.62","7,552.54"
1,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,M,Black/African American,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"25,370.86","3,469.55"
2,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,F,Other Race,Spanish/Hispanic,...,Minor,Minor,Medical,Medicaid,NaN,NaN,NaN,N,"23,876.78","6,180.33"
3,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,100,F,Black/African American,Not Span/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"43,319.05","12,588.93"
4,New York City,Bronx,7000006.0,1168.0,Montefiore Medical Center-Wakefield Hospital,18 to 29,104,M,Other Race,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,"40,266.23","10,355.99"


In [3]:
import json

with open("../data/data_info.json", "r") as f:
    data_info = json.load(f)

In [4]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

Hospital Service Area 5390
Hospital County 5390
Operating Certificate Number 5961
Permanent Facility Id 5390
Zip Code - 3 digits 41227
CCSR Procedure Code 582815
CCSR Procedure Description 582815
APR Severity of Illness Description 636
APR Risk of Mortality 636
Payment Typology 2 1121497
Payment Typology 3 1814362
Birth Weight 1894796


In [5]:
# Handle missing data and changing column type
for col_name, info in data_info.items():
    print(col_name)
    if info["type"] == "int64":
        if col_name == "Zip Code - 3 digits":
            data[col_name] = data[col_name].replace("OOS", 000)
            data[col_name] = data[col_name].fillna(000)
        if col_name == "Length of Stay":
            data[col_name] = data[col_name].replace("120 +", 121)
        if col_name == "Birth Weight":
            data[col_name] = data[col_name].replace("UNKN", -1)
            data[col_name] = data[col_name].fillna(-1)
        else:
            data[col_name] = data[col_name].fillna(-1)
            
        data[col_name] = pd.to_numeric(data[col_name]).astype('int')
        
    elif info["type"] == "str":
        data[col_name] = data[col_name].fillna("Unknown")
        data[col_name] = data[col_name].astype(str)

    elif info["type"] == "float64":
        data[col_name] = data[col_name].str.replace(',', '')
        data[col_name] = pd.to_numeric(data[col_name]).astype('float')


Hospital Service Area
Hospital County
Operating Certificate Number
Permanent Facility Id
Facility Name
Age Group
Zip Code - 3 digits
Gender
Race
Ethnicity
Length of Stay
Type of Admission
Patient Disposition
Discharge Year
CCSR Diagnosis Code
CCSR Diagnosis Description
CCSR Procedure Code
CCSR Procedure Description
APR DRG Code
APR DRG Description
APR MDC Code
APR MDC Description
APR Severity of Illness Code
APR Severity of Illness Description
APR Risk of Mortality
APR Medical Surgical Description
Payment Typology 1
Payment Typology 2
Payment Typology 3
Birth Weight
Emergency Department Indicator
Total Charges
Total Costs


In [6]:
for col in data.columns:
    if data[col].isna().any():
        print(col, data[col].isna().sum())

In [7]:
for col in data.columns:
    print(col, data[col].dtype)

Hospital Service Area object
Hospital County object
Operating Certificate Number int64
Permanent Facility Id int64
Facility Name object
Age Group object
Zip Code - 3 digits int64
Gender object
Race object
Ethnicity object
Length of Stay int64
Type of Admission object
Patient Disposition object
Discharge Year int64
CCSR Diagnosis Code object
CCSR Diagnosis Description object
CCSR Procedure Code object
CCSR Procedure Description object
APR DRG Code int64
APR DRG Description object
APR MDC Code int64
APR MDC Description object
APR Severity of Illness Code object
APR Severity of Illness Description object
APR Risk of Mortality object
APR Medical Surgical Description object
Payment Typology 1 object
Payment Typology 2 object
Payment Typology 3 object
Birth Weight int64
Emergency Department Indicator object
Total Charges float64
Total Costs float64


In [8]:
# Identify categorical columns and convert them to categories
categorical_columns = []
for col in data.columns:
    if data[col].dtype == "object":
        data[col] = data[col].astype('category')

In [38]:
model = joblib.load("../models/lgb_model.pkl")

In [15]:
# load shap explainer
explainer = joblib.load('../models/shap_explainer.pkl')

In [26]:
data_info["CCSR Procedure Description"]["options"] = []
data_info["APR DRG Description"]["options"] = []

In [54]:
from langchain.agents import Tool
from langchain.tools import StructuredTool
from langchain_experimental.plan_and_execute import PlanAndExecute, load_chat_planner, load_agent_executor
from langchain_google_genai import ChatGoogleGenerativeAI
import pandas as pd
import os
import json


patient_dict = {
    "Hospital Service Area": "New York City",
    "Hospital County": "Kings",
    "Operating Certificate Number": 7001009,
    "Permanent Facility Id": 1294,
    "Facility Name": "Coney Island Hospital",
    "Age Group": "18 to 29",
    "Zip Code - 3 digits": 112,
    "Gender": "F",
    "Race": "Black/African American",
    "Ethnicity": "Not Span/Hispanic",
    "Length of Stay": 1,
    "Type of Admission": "Emergency",
    "Patient Disposition": "Home or Self Care",
    "Discharge Year": 2022,
    "CCSR Diagnosis Code": "INF012",
    "CCSR Diagnosis Description": "COVID-19",
    "CCSR Procedure Code": "ADM015",
    "CCSR Procedure Description": "ADMINISTRATION OF ANTIBIOTICS",
    "APR DRG Code": 137,
    "APR DRG Description": "MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS",
    "APR MDC Code": 4,
    "APR MDC Description": "DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM",
    "APR Severity of Illness Code": 3,
    "APR Severity of Illness Description": "Major",
    "APR Risk of Mortality": "Moderate",
    "APR Medical Surgical Description": "Medical",
    "Payment Typology 1": "Medicaid",
    "Payment Typology 2": "Unknown",
    "Payment Typology 3": "Unknown",
    "Birth Weight": -1,
    "Emergency Department Indicator": "Y",
    "Total Charges": 8803.14
}

shared_memory = {}
explainer = joblib.load('../models/shap_explainer.pkl')


def extract_shap_info():
    patient_df = pd.DataFrame(patient_dict, index=[0])
    for col in data.select_dtypes(['category']).columns:
        patient_df[col] = pd.Categorical(patient_df[col], categories=data[col].cat.categories)

    shared_memory["current cost"] = model.predict(patient_df).item()
    shap_values = explainer(patient_df)
    shap_info = {}
    for i, col in enumerate(patient_df.columns):
        shap_info[col] = {
            "type": data_info[col]["type"],
            "value": shap_values.data[0][i],
            "shap value": shap_values.values[0][i],
            "other options": data_info[col].get("options", [])
        }
    return json.dumps(shap_info)


def suggest_strategies(shap_info: str):
    shap_info = json.dumps(shap_info) if isinstance(shap_info, dict) else shap_info
    prompt = f"""
    You are a healthcare cost-reduction expert. You are given structured data for a single inpatient case.

    Each feature contains:
    - "type": the data type
    - "value": the observed value
    - "shap value": contribution to the total inpatient cost
    - "other options": the possible values for that feature (if any)

    Your tasks:
    1. For each feature, suggest a specific, feasible strategy to reduce its cost impact, using the "other options" where applicable to that patient. If there is no strategy, leave this empty.
    2. Conclude with a short summary (2–3 sentences) explaining the overall cost-reduction logic you applied.
    3. Return your response in this JSON format only (no additional text):

    {{
    "Feature Name 1": "strategy",
    "Feature Name 2": "strategy",
    ...
    "Summary": "your overall summary"
    }}

    Data:
    ###patient_info###
    """
    prompt = prompt.replace("###patient_info###", shap_info)
    strategies = llm.invoke(prompt).content.strip()
    return json.dumps(strategies)


def cost_predition(strategies: dict):
    strategies = json.loads(strategies) if isinstance(strategies, str) else strategies
    min_cost = shared_memory["current cost"]
    for col, method in strategies.items():
        if col == "Summary":
            continue
        info = data_info[col]
        if method != None and len(method) > 5:
            if info["type"] == "str" and "other options" in info:
                for option in info["other options"]:
                    updated_patient_dict = patient_dict.copy()
                    # apply changes
                    updated_patient_dict[col] = option
                    updated_patient_df = pd.DataFrame(updated_patient_dict, index=[0])
                    for col in data.select_dtypes(['category']).columns:
                        updated_patient_df[col] = pd.Categorical(updated_patient_df[col], categories=data[col].cat.categories)
                    cost = model.predict(updated_patient_df)
                    if cost <= min_cost:
                        min_cost = cost
                        shared_memory["new cost"] = min_cost
                        shared_memory["target feature"] = col
                        shared_memory["strategy"] = option
    return json.dumps(shared_memory)

            
# --- LLM Setup ---
gemini_api_key = os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=gemini_api_key)

# --- Tool Collection and Agent Setup ---
tools = [
    StructuredTool.from_function(name="ExtractSHAPInfo", func=extract_shap_info, description="Compute SHAP values and return attributions"),
    StructuredTool.from_function(name="SuggestStrategies", func=suggest_strategies, description="Suggest realistic cost-reducing strategies for given features"),
    StructuredTool.from_function(name="CostPredition", func=cost_predition, description="Predict the cost after applying the strategies", return_direct=True),
]


planner = load_chat_planner(llm)
executor = load_agent_executor(llm=llm, tools=tools, verbose=False)
agent = PlanAndExecute(planner=planner, executor=executor, verbose=False, input_key="input")

# --- Execute Instruction ---
if __name__ == "__main__":
    prompt = f"""
        You are an expert in healthcare cost optimization and hospital operations strategy.

        You will receive structured data for a single inpatient hospital case. Each feature includes:
        - "type": the data type (e.g., int, float, str)
        - "value": the observed value for the case
        - "shap value": contribution to total inpatient cost
        - "other options": possible alternative values (if any)

        ### OBJECTIVE ###
        1. Identify the most significant cost-driving features based on SHAP values.
        2. Suggest specific, feasible strategies to reduce cost impact, using "other options" when appropriate. Leave blank if no strategy applies.
        3. Summarize the cost-reduction logic in 2–3 sentences.

        ### PLAN ###
        Execute these steps using the available tools:
        1. Run `ExtractSHAPInfo` with the provided patient dictionary.
        2. Run `SuggestStrategies` with the result.
        3. Run `CostPredition` with suggested strategies, return the results and STOP.
        """
    
    max_retries = 5
    for attempt in range(max_retries):
        try:
            output = agent.run({"input": prompt})
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
    else:
        print("All retries failed.")

    print("Success!")
    print(output)


Attempt 1 failed: 'age'
Success!
The current cost is $4451.42 after applying the suggested cost reduction strategies.
